# 인도 장애물 YOLO26n 학습 (Colab)

AI Hub [인도보행 영상](https://www.aihub.or.kr/aihubdata/data/view.do?dataSetSn=189)의 Bounding Box 데이터와 Hugging Face [ROD-Dataset](https://huggingface.co/datasets/Abtinzandi/Obstacle-Detection-Dataset-YOLO)으로 YOLO26n 을 파인튜닝하고 LiteRT(`.tflite`)로 내보낸다.

원본 라벨(AI Hub·ROD)은 안내 방식이 같은 것끼리 그룹(사람·이륜차·차·장애물·기둥·계단)으로 묶어 학습한다. 박스는 그대로 두고 클래스 번호만 바꾸므로 다시 라벨링하지 않는다.

**처음 한 번 준비**
1. AI Hub 에서 인도보행 영상 데이터 사용 신청 → 승인
2. AI Hub > AI 개발지원 > 오픈 API 에서 API key 발급 (메일로 옴)
3. Colab 왼쪽 열쇠 아이콘(Secrets)에 `AIHUB_API_KEY` 로 넣고 노트북 접근 허용. **키를 코드에 쓰지 않는다.**
4. 런타임 > 런타임 유형 변경 > **T4 GPU**

**흐름**: 다운로드(Colab 디스크) → YOLO 형식 변환·640px 축소 → 데이터셋을 Drive 에 보관 → 학습(결과는 Drive) → `.tflite` 내보내기

세션이 끊기면 1번 설정 셀부터 다시 실행한다. Drive 에 데이터셋 zip 이 있으면 다운로드·변환을 건너뛰고, 학습은 `RESUME = True` 로 이어서 한다.

## 1. 설정

In [ ]:
from pathlib import Path

# AI Hub 인도보행 영상
DATASET_KEY = "189"
# 받을 Bbox 파일 key. `2. 파일 목록` 셀 결과에서 골라 넣는다. 하나(약 10GB)로 시작한다.
FILE_KEYS = ["49933"]  # Bbox_10_new.zip

# 학습할 그룹. 순서가 곧 클래스 번호이므로 학습 뒤에는 바꾸지 않는다.
CLASSES = [
    "person",  # 사람: 멈춤·다가옴·빈 방향 계산
    "two_wheeler",  # 자전거·킥보드·오토바이: 빠르게 움직임
    "car",  # 차: 이면도로·인도 주차
    "obstacle",  # 바닥에 놓인 장애물: 볼라드·라바콘·벤치 등
    "pole",  # 가늘고 높은 것: 기둥·나무줄기·표지판
    "stairs",  # 계단: 넘어질 위험이 커서 따로 둔다
    # "ped_signal_red",  # 빨간불 데이터를 구하면 맨 뒤에 추가한다
]

# AI Hub 라벨 이름(ROD 는 ROD_NAME_MAP 으로 바꾼 이름) → 그룹.
# 여기 없는 라벨의 박스는 버린다.
# 박스는 그대로 두고 번호만 바꾸므로 다시 라벨링하지 않는다.
LABEL_TO_CLASS = {
    # 사람
    "person": "person",
    # 이륜차
    "bicycle": "two_wheeler",
    "motorcycle": "two_wheeler",
    "scooter": "two_wheeler",
    # 차 (AI Hub 라벨 이름은 받은 데이터에서 확인한다)
    "car": "car",
    "bus": "car",
    "truck": "car",
    # 바닥에 놓인 장애물
    "bollard": "obstacle",
    "barricade": "obstacle",
    "movable_signage": "obstacle",
    "traffic_cone": "obstacle",
    "traffic_barrel": "obstacle",
    "guard_rail": "obstacle",
    "bench": "obstacle",
    "chair": "obstacle",
    "table": "obstacle",
    "potted_plant": "obstacle",
    "fire_hydrant": "obstacle",
    "power_controller": "obstacle",
    "kiosk": "obstacle",
    "dustbin": "obstacle",
    "bicycle_rack": "obstacle",
    # 사람이 밀거나 끄는 것. 미는 사람은 person 으로 따로 잡힌다
    "stroller": "obstacle",
    "wheelchair": "obstacle",
    "carrier": "obstacle",
    # 가늘고 높은 것
    "pole": "pole",
    "tree_trunk": "pole",
    "parking_meter": "pole",
    "stop": "pole",
    # 계단
    "stairs": "stairs",
    # manhole: 바닥과 높이가 같아 장애물이 아니므로 버린다
}
assert set(LABEL_TO_CLASS.values()) <= set(CLASSES)

# ROD-Dataset (Hugging Face, MIT): 보행자 시점 장애물 25종, YOLO 형식
USE_ROD = True
ROD_REPO = "Abtinzandi/Obstacle-Detection-Dataset-YOLO"
# ROD 클래스 → AI Hub 라벨 이름. None 이거나 LABEL_TO_CLASS 에 없으면 버린다.
ROD_NAME_MAP = {
    "Bike": "bicycle",
    "Motorcycle": "motorcycle",
    "Person": "person",
    "Electrical Pole": "pole",
    # ROD 의 Tree 는 나무 전체 박스라 AI Hub tree_trunk(기둥만)와 맞지 않아 버린다
    "Tree": None,
    "Bench": "bench",
    "Chair": "chair",
    "Plant Pot": "potted_plant",
    "Fire hydrant": "fire_hydrant",
    "Electrical Box": "power_controller",
    "Stairs": "stairs",
    "Manhole": "manhole",
    "Traffic Cone": "traffic_cone",
    "Teraffic Barrel": "traffic_barrel",  # ROD 원본 철자 그대로
    "Dustbin": "dustbin",
    "Guard rail": "guard_rail",
    "Bicycle Rack": "bicycle_rack",
    # 차는 LABEL_TO_CLASS 에서 car 그룹으로 묶는다
    "Car": "car",
    "Truck": "truck",
    "Bus": "bus",
    # 장애물이 아니거나 높은 곳에 있는 것
    "Building": None,
    "Road": None,
    "Pedestrian crosswalk": None,
    "Traffic sign": None,
    "Dog": None,
}

IMG_SIZE = 640  # 학습 입력 크기. 이미지도 긴 변을 이 크기로 줄여 저장한다
VAL_RATIO = 0.1  # 검증셋 비율 (XML 묶음 단위)
EPOCHS = 100
BATCH = 16
RUN_NAME = "sidewalk_v1"
RESUME = False  # 세션이 끊겨 이어서 학습할 때 True

RAW_DIR = Path("/content/raw")  # 원본(세션 끝나면 사라짐)
ROD_DIR = Path("/content/rod")
DATA_DIR = Path("/content/sidewalk_yolo")  # 변환 결과
DRIVE_DIR = Path("/content/drive/MyDrive/saitgil")
ZIP_NAME = "_".join(FILE_KEYS) + ("_rod" if USE_ROD else "")
DATASET_ZIP = DRIVE_DIR / f"sidewalk_yolo_{ZIP_NAME}.zip"
RUNS_DIR = DRIVE_DIR / "runs"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics

from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## 2. AI Hub 에서 받기

Drive 에 변환된 데이터셋이 있으면 이 단계와 3단계를 건너뛰고 4단계로 간다.

In [ ]:
import os

from google.colab import userdata

# 키는 Secrets 에서 읽어 환경변수로만 넘긴다(출력하지 않음).
os.environ["AIHUB_API_KEY"] = userdata.get("AIHUB_API_KEY")

!curl -s -o /usr/local/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do
!chmod +x /usr/local/bin/aihubshell

In [ ]:
# 파일 목록: Bbox_xx_new.zip 줄 끝 숫자가 filekey 다.
!aihubshell -mode l -datasetkey {DATASET_KEY} | grep -i -E "bbox|라벨|xml"

In [ ]:
DOWNLOAD_ARGS = f"-mode d -datasetkey {DATASET_KEY} -filekey {','.join(FILE_KEYS)}"
# 다운로드가 끝나면 aihubshell 이 압축을 풀고 zip 을 지운다.
# 받는 용량의 2~3배 디스크가 필요하다.
if DATASET_ZIP.exists():
    print("Drive 에 데이터셋이 있어 받지 않습니다:", DATASET_ZIP)
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    %cd {RAW_DIR}
    !aihubshell {DOWNLOAD_ARGS} -aihubapikey "$$AIHUB_API_KEY"
    %cd /content
    !df -h /content | tail -1
    !find {RAW_DIR} -name "*.xml" | head -3
    !find {RAW_DIR} -name "*.xml" | wc -l

## 3. YOLO 형식으로 바꾸기

AI Hub CVAT XML 의 `box`(px 좌표)를 YOLO `class cx cy w h`(0~1)로 바꾸고, 이미지는 긴 변 640px JPEG 로 줄인다. ROD 는 이미 YOLO 형식이라 클래스 번호만 바꾼다. 라벨 이름은 `LABEL_TO_CLASS` 로 그룹 번호가 되고, 매핑에 없는 라벨은 버린다.

In [ ]:
import random
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

from PIL import Image

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def index_images(root: Path) -> dict[str, Path]:
    """파일 이름 → 경로.

    XML 의 image name 은 폴더가 빠져 있을 수 있어 이름으로 찾는다.
    """
    return {p.name: p for p in root.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES}


def yolo_lines(image: ET.Element, class_ids: dict[str, int]) -> list[str]:
    """CVAT box(xtl, ytl, xbr, ybr, px) → YOLO 'class cx cy w h' (0~1)."""
    width, height = float(image.get("width")), float(image.get("height"))
    lines = []
    for box in image.iter("box"):
        label = box.get("label")
        if label not in class_ids:
            continue
        xtl, ytl = max(float(box.get("xtl")), 0), max(float(box.get("ytl")), 0)
        xbr, ybr = min(float(box.get("xbr")), width), min(float(box.get("ybr")), height)
        if xbr <= xtl or ybr <= ytl:
            continue
        cx, cy = (xtl + xbr) / 2 / width, (ytl + ybr) / 2 / height
        w, h = (xbr - xtl) / width, (ybr - ytl) / height
        lines.append(f"{class_ids[label]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines


def save_resized(src: Path, dst: Path, long_side: int) -> None:
    """긴 변을 long_side 로 줄여 JPEG 로 저장한다. 라벨은 0~1 비율이라 그대로 맞는다."""
    with Image.open(src) as img:
        img = img.convert("RGB")
        img.thumbnail((long_side, long_side))
        img.save(dst, "JPEG", quality=90)


def convert(
    raw_dir: Path,
    out_dir: Path,
    classes: list[str],
    label_to_class: dict[str, str],
    long_side: int,
    val_ratio: float,
    seed: int = 0,
) -> Counter:
    """AI Hub 인도보행 Bbox(CVAT XML) → YOLO 데이터셋.

    같은 XML(같은 촬영 묶음)의 사진은 거의 같아,
    검증셋은 사진이 아니라 XML 단위로 나눈다.
    라벨 이름은 label_to_class 로 그룹 번호로 바꾸고, 없는 라벨은 버린다.
    """
    class_ids = {label: classes.index(group) for label, group in label_to_class.items()}
    images = index_images(raw_dir)
    xmls = sorted(raw_dir.rglob("*.xml"))
    random.Random(seed).shuffle(xmls)
    n_val = max(1, round(len(xmls) * val_ratio)) if len(xmls) > 1 else 0
    counts: Counter = Counter()
    for i, xml_path in enumerate(xmls):
        split = "val" if i < n_val else "train"
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        for image in ET.parse(xml_path).getroot().iter("image"):
            src = images.get(Path(image.get("name")).name)
            if src is None:
                counts["missing_image"] += 1
                continue
            lines = yolo_lines(image, class_ids)
            # 이름이 겹치지 않게 XML 이름을 앞에 붙인다.
            stem = f"{xml_path.stem}_{src.stem}"
            save_resized(src, out_dir / "images" / split / f"{stem}.jpg", long_side)
            (out_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
            counts[f"{split}_images"] += 1
            counts.update(classes[int(line.split()[0])] for line in lines)
    (out_dir / "data.yaml").write_text(
        f"path: {out_dir}\ntrain: images/train\nval: images/val\n"
        + "names:\n"
        + "".join(f"  {i}: {name}\n" for i, name in enumerate(classes))
    )
    return counts


def add_yolo_dataset(
    src_dir: Path,
    out_dir: Path,
    src_names: list[str],
    name_map: dict[str, str | None],
    classes: list[str],
    label_to_class: dict[str, str],
    long_side: int,
    prefix: str,
) -> Counter:
    """이미 YOLO 형식인 데이터셋(train/valid)을 클래스 번호를 바꿔 out_dir 에 더한다.

    name_map 으로 AI Hub 라벨 이름으로 바꾼 뒤 label_to_class 로 그룹 번호를 정한다.
    name_map 에 없거나 None 이거나 label_to_class 에 없는 클래스의 박스는 버린다.
    """
    class_ids = {label: classes.index(group) for label, group in label_to_class.items()}
    remap = {
        i: class_ids[name_map[name]]
        for i, name in enumerate(src_names)
        if name_map.get(name) in class_ids
    }
    counts: Counter = Counter()
    for src_split, split in [("train", "train"), ("valid", "val")]:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        for src in sorted((src_dir / src_split / "images").iterdir()):
            if src.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            label = src_dir / src_split / "labels" / f"{src.stem}.txt"
            lines = []
            if label.exists():
                for line in label.read_text().splitlines():
                    parts = line.split()
                    # 박스(5칸)만 쓴다. 폴리곤 줄은 버린다.
                    if len(parts) == 5 and int(parts[0]) in remap:
                        lines.append(" ".join([str(remap[int(parts[0])]), *parts[1:]]))
            stem = f"{prefix}_{src.stem}"
            save_resized(src, out_dir / "images" / split / f"{stem}.jpg", long_side)
            (out_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
            counts[f"{split}_images"] += 1
            counts.update(classes[int(line.split()[0])] for line in lines)
    return counts

In [ ]:
if DATASET_ZIP.exists():
    print("Drive 에 데이터셋이 있어 변환하지 않습니다.")
else:
    counts = convert(RAW_DIR, DATA_DIR, CLASSES, LABEL_TO_CLASS, IMG_SIZE, VAL_RATIO)
    for key in ["train_images", "val_images", "missing_image"]:
        print(key, counts.pop(key, 0))
    print("AI Hub 그룹별 박스 수:", dict(counts.most_common()))
    # 원본은 지워 디스크를 비운다.
    !rm -rf {RAW_DIR}

### ROD 더하기

ROD-Dataset(약 1.5GB)을 받아 AI Hub 결과에 더한다. 파일 이름 앞에 `rod_` 를 붙인다.

ROD 는 공개 데이터 26개를 합친 것이라, 한 출처가 라벨링하지 않은 물체(예: 맨홀 사진 속 사람)는 박스가 없을 수 있다. 팀 모델을 합칠 때 자동 라벨로 채운다.

In [ ]:
import yaml
from huggingface_hub import snapshot_download

if DATASET_ZIP.exists() or not USE_ROD:
    print("ROD 를 더하지 않습니다.")
else:
    snapshot_download(ROD_REPO, repo_type="dataset", local_dir=ROD_DIR)
    rod_names = yaml.safe_load((ROD_DIR / "data.yaml").read_text())["names"]
    counts = add_yolo_dataset(
        ROD_DIR,
        DATA_DIR,
        rod_names,
        ROD_NAME_MAP,
        CLASSES,
        LABEL_TO_CLASS,
        IMG_SIZE,
        "rod",
    )
    for key in ["train_images", "val_images"]:
        print(key, counts.pop(key, 0))
    print("ROD 그룹별 박스 수:", dict(counts.most_common()))
    !rm -rf {ROD_DIR}

### Drive 에 보관

In [ ]:
if DATASET_ZIP.exists():
    print("이미 있습니다:", DATASET_ZIP)
else:
    # 줄인 데이터셋을 Drive 에 보관한다.
    !cd /content && zip -q -r {DATASET_ZIP} {DATA_DIR.name}
    !ls -lh {DATASET_ZIP}

## 4. 데이터셋 준비 (Drive 에서 복원)

In [ ]:
if not (DATA_DIR / "data.yaml").exists():
    !cd /content && unzip -q {DATASET_ZIP}
# data.yaml 의 path 를 지금 위치로 맞춘다.
yaml_path = DATA_DIR / "data.yaml"
lines = yaml_path.read_text().splitlines()
yaml_path.write_text("\n".join([f"path: {DATA_DIR}"] + lines[1:]) + "\n")
print(yaml_path.read_text())

## 5. 학습

결과(`best.pt`, `last.pt`, 그래프)는 Drive `saitgil/runs/` 에 쌓인다. 세션이 끊기면 `RESUME = True` 로 바꾸고 1·4·5 셀을 다시 실행한다.

In [ ]:
from ultralytics import YOLO

last = RUNS_DIR / RUN_NAME / "weights" / "last.pt"
if RESUME and last.exists():
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO("yolo26n.pt")
    model.train(
        data=str(DATA_DIR / "data.yaml"),
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        patience=20,
    )

## 6. 검증

In [ ]:
best = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
metrics = YOLO(str(best)).val(data=str(DATA_DIR / "data.yaml"), imgsz=IMG_SIZE)
print("mAP50:", round(metrics.box.map50, 3), "mAP50-95:", round(metrics.box.map, 3))
# ap50 은 검증셋에 나온 클래스만 ap_class_index 순서로 담긴다.
for class_id, ap50 in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"{CLASSES[class_id]:26s} AP50 {ap50:.3f}")

## 7. LiteRT(`.tflite`)로 내보내기

받은 `.tflite` 는 커밋하지 않고 앱 assets(`frontend/android/app/src/main/assets/models/`)에 복사한다.

In [ ]:
exported = YOLO(str(best)).export(format="tflite", imgsz=IMG_SIZE)
print(exported)
!cp -r {Path(exported).parent} {RUNS_DIR / RUN_NAME}/
!ls -lh {RUNS_DIR / RUN_NAME}